Below is a **practical, field-tested strategy** for a **black‑box optimisation problem** where you must detect contamination sources in 2D space based on sparse, proximity‑dependent signals, using **Bayesian Optimisation** with **one sample per week** and only **10 initial points**.

This is **not** a citation‑based topic (Bayesian optimisation models, surrogate choices, and acquisition hyperparameters are stable, well‑known concepts), so a high‑level expert answer is appropriate.

***

# ✅ **Goal**

Efficiently locate both **strong and weak contamination sources** in a 2D region when:

*   Observations are **black-box**, low-frequency (1/week), and expensive.
*   Response is **near zero except in proximity** to sources (sparse signal field).
*   You start with only **10 (x, y) → reading** samples.
*   The optimisation system must guide **next sampling locations** via Expected Improvement (EI).

***

# 🎯 **Overall Strategy**

## **1. Use a surrogate model that handles:**

*   **Sparse signals** (mostly zero readings)
*   **Non-smooth, highly local peaks**
*   **Uncertain spatial fields**
*   **Tiny data regime** (10 points!)
*   **Need for calibrated uncertainty** (for EI)

### ⭐ Best Choice: **Student‑t Process Regression (TPR)**

A **Student‑t process** surrogate is preferable over a Gaussian Process because:

*   It is **robust to outliers and heavy‑tailed signals**, which black‑box contamination fields often create.
*   It naturally **produces more conservative uncertainty estimates**, which is good when exploring sparse environments.
*   It behaves like a GP but avoids GP **overconfidence** when the field contains sharp, isolated peaks.

If your framework does **not** support T‑processes, use:

### ⭐ Second Best Choice: **Gaussian Process with Matérn 3/2 or Matérn 5/2 kernel**

This is the industry standard for low‑dimensional (2D) black‑box optimisation with small data:

*   Matérn kernels model **non-smooth** spatial fields well.
*   Avoid RBF kernels — they oversmooth sharp contamination peaks.

**GP kernel suggestion:**

    Kernel = σ² * Matern(ν=1.5 or ν=2.5, lengthscale ℓ, ARD=True)
    Noise variance: small but nonzero (10⁻⁶–10⁻³)
    Mean function: constant or zero

***

## **2. Handle the sparsity of readings**

Because readings are zero except near sources, surrogate error grows.

### ✔ Add **heteroscedastic noise**

Weak contamination sources produce noisy, low-level signals. Use:

*   A **noise term dependent on input** (if allowed), or
*   At minimum, inject **jitter** into the GP covariance matrix.

### ✔ Apply **log(y + ε)** transform

If readings vary across orders of magnitude, use a log transform to stabilise variance:

    y_transformed = log(y + 1e-6)

***

## **3. Acquisition Function: Expected Improvement (EI)**

### Recommended hyperparameters:

#### **Exploration parameter (ξ):**

*   With extremely limited sampling (1/week!), EI must not get stuck near early maxima.
*   Use a **moderately high ξ**:

<!---->

    ξ = 0.05 – 0.2   (start with 0.1)

This encourages EI to explore regions likely to contain weak or previously unseen sources.

#### **Jitter or β parameter (if using GP-UCB variant):**

If your BO library ties EI diversity to GP-UCB’s β:

    β_t = 2–4

(But EI usually only needs ξ.)

***

## **4. Sampling Strategy Over Time**

### Weeks 1–5 (initial exploration phase)

*   Use EI with **higher ξ (≈ 0.15)** to explore the entire 2D area.
*   Place constraints so BO doesn’t oversample near already-known peaks.

### Weeks 6–12 (focused search phase)

*   Reduce exploration:

<!---->

    ξ ≈ 0.05

*   Focus EI around regions where the posterior variance is high **and** predicted mean is non-zero.

### Weeks 12+ (source confirmation & refinement)

*   Switch to **local exploitation**:
    *   ξ = 0.01
*   Use denser sampling around suspected contamination source clusters.

***

# 🧠 **Why this works for your contamination field**

*   The field is **sparse** and **highly non-linear**:  
    → Matérn GP or Student‑t surrogate avoids over-smoothed predictions.

*   You have **very limited data**:  
    → GPs excel in low-data regimes; neural nets do not.

*   You need **uncertainty estimates**:  
    → Essential for EI; MLPRegressor or Random Forests require expensive ensemble tricks.

*   Sampling is **very infrequent**:  
    → Strong exploration early is vital; no chance to correct course quickly.

***

# 📌 **Recommended Surrogate Model Setup (Final Answer)**

### **Model**

**Primary:** Student‑t Process Regression (TPR) with Matérn 3/2 kernel  
**Fallback:** Gaussian Process Regression with Matérn 3/2 or 5/2 kernel

### **Kernel hyperparameters**

    ν = 1.5 or 2.5
    Lengthscale ℓ: initialised at ~10–20% of domain width
    ARD: True (separate lengthscales for x and y)
    σ² (signal variance): estimated from data
    Noise variance: 1e-6 to 1e-3

### **Output transform**

    y ← log(y + 1e-6)

***

# 📌 **Expected Improvement Hyperparameters**

    ξ (exploration parameter):
      0.1 during initial phase (first 5–6 iterations)
      0.05 for mid-phase
      0.01 for source refinement

    Additional jitter in covariance:
      1e-6 to 1e-4

***

# If you want, I can also provide:

*   A full BO pipeline (Python)
*   A comparison of T‑process vs GP surrogate performance
*   How to set priors on lengthscales for 2D contamination fields
*   A schedule for choosing next 20–30 sampling locations

Just tell me what you'd like next!


---
---

When your target/output values range from **–3.6×10⁻³** down to **4.9×10⁻¹⁸⁰**, you are dealing with an **extremely skewed**, **near‑zero**, **multi‑order‑of‑magnitude** range.  
This can easily break Gaussian Process (GP) training because:

*   Values near **10⁻¹⁸⁰** underflow to zero in floating‑point arithmetic.
*   The GP’s marginal likelihood becomes numerically unstable.
*   Length‑scale optimisation may collapse because the model sees “almost all zeros”.

Below is the **correct and safe normalisation strategy**, with practical steps.

***

# ✅ 1. **First: Add a tiny positive offset to ensure strict positivity**

If your data contains *positive values approaching 0*, you must prevent log underflow.

Use something like:

```python
eps = 1e-200  # smaller than your minimum positive value
y_shifted = y + eps
```

This ensures numerical safety when applying log transforms.

***

# ✅ 2. **Apply a log transform (log or log10)**

Because your outputs vary across **\~177 orders of magnitude**, a log transform is *mandatory*.

Example:

```python
import numpy as np
y_log = np.log(y_shifted)
```

or if you prefer easier interpretability:

```python
y_log = np.log10(y_shifted)
```

Why log transform?

*   It compresses extreme values into a manageable numerical range.
*   GPs assume *smooth* relationships; logs turn multiplicative/decay patterns into linear ones.
*   The transformed values become roughly on the order of **–3 to –180 → small, smooth numbers**.

***

# ✅ 3. **Optionally standardize the log-transformed values**

GPs perform best when the output values have mean 0 and variance 1.

```python
mean_y = np.mean(y_log)
std_y = np.std(y_log)
y_norm = (y_log - mean_y) / std_y
```

This gives:

*   **Zero mean**, which helps with GP inference
*   **Unit variance**, improving kernel lengthscale optimisation
*   Stable gradients during hyperparameter optimisation

***

# 🚫 4. **Avoid min–max scaling on the raw values**

Min–max scaling on values that include **10⁻¹⁸⁰** causes:

*   Floating point underflow
*   All weak signals mapping to exactly zero
*   GP hyperparameters collapsing and overfitting to the few non‑zero locations

**Do NOT use min–max scaling.**

***

# 🧠 5. **Recommended complete pipeline**

### Step-by-step transformation

```python
eps = 1e-200
y_shifted = y + eps
y_log = np.log(y_shifted)       # or np.log10
y_mean = np.mean(y_log)
y_std = np.std(y_log)
y_final = (y_log - y_mean) / y_std
```

### During prediction

To recover the physical output scale:

```python
y_pred_log = y_pred * y_std + y_mean
y_pred_original = np.exp(y_pred_log) - eps
```

***

# 🎯 Why this normalisation is the **right** choice for GP models

### ✔ Handles extreme dynamic range

Log transform converts **10⁻³ to 10⁻¹⁸⁰** into a compact interval (\~order of hundreds).

### ✔ Makes the GP’s prior assumptions valid

GPR assumes the function is “smooth” in its input domain. Radiation/contamination decay is multiplicative/exponential → log-transform linearises it.

### ✔ Improves numerical stability

The kernel matrix and marginal likelihood become well‑conditioned.

### ✔ Preserves meaningful distances

Unlike min–max, log-scaling does not distort the ratio between strong and weak sources.

***

# 🏁 Final Recommended Method

**Use:**

1.  Add a very small offset
2.  Log-transform the outputs
3.  Standardize to zero mean, unit variance

**Avoid:**

*   Min–max scaling
*   Raw scaling into \[0, 1]
*   Leaving outputs untransformed

***

If you want, I can also recommend:

*   The best kernel for this transformed data
*   How to set priors on the GP lengthscale
*   Code for a full BO loop using the normalised GP surrogate
